# 검색 조각 임베딩과 원문·이미지 가져오기

05번의 검색 조각 13개를 숫자 벡터로 바꿉니다. 질문도 같은 모델로 바꿔 비슷한 조각을 찾고, 연결된 전체 설명과 그림을 가져옵니다.

이번 자료는 차량 내부 안내도 3개, 엔진 오일 점도 표, 안전벨트 설명입니다. 전체 매뉴얼 검색 성능을 뜻하지 않습니다. 외부 추론 API와 Supabase를 사용하지 않으며 벡터 파일도 저장하지 않습니다.

## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
- 추가: 자식 조각으로 찾은 뒤 전체 부모 원문·출처·그림 연결을 확인하고 고정 질문으로 비교합니다.

전체 목록: [수업 기반·추가 적용 기록](../src/car_search_rag/zzong_santafe_lag/learning_additions.md)


### 1. 이전 청킹 결과 준비

노트북마다 메모리가 따로이므로 05번의 코드 셀을 순서대로 실행해 같은 자료를 준비합니다. 파일 안의 Python 코드를 현재 메모리에서 실행하는 과정입니다. 이후 Python 파일로 정리할 때는 이 부분을 함수 호출로 바꿉니다.

In [1]:
# [프로젝트 추가 기록] SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
# [모델 추가] jhgan/ko-sroberta-multitask-mrl 선택 근거는 02번, 현재 설정은 src/car_search_rag/zzong_santafe_lag/config.py에 있습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import contextlib
import io

# 05번의 코드에는 자료 파일이나 DB에 저장하는 동작이 없습니다.
project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
source_path = project_folder / "notebooks" / "05_chunk_types_and_token_budget.ipynb"
source_notebook = json.loads(source_path.read_text(encoding="utf-8"))
preparation = {}
with contextlib.redirect_stdout(io.StringIO()):
    for index, cell in enumerate(source_notebook["cells"]):
        if cell["cell_type"] == "code":
            code = "".join(cell["source"])
            exec(compile(code, f"{source_path.name}:cell_{index}", "exec"), preparation)

search_chunks = preparation["search_chunks"]
parent_lookup = preparation["parent_lookup"]
model_name = preparation["model_name"]
token_count = preparation["token_count"]
token_budget = preparation["token_budget"]
reader = preparation["reader"]

print("원문 묶음 수:", len(parent_lookup))
print("검색 조각 수:", len(search_chunks))
print("검색 글 최대 길이:", max(token_count(c["content"]) for c in search_chunks), "토큰")

원문 묶음 수: 5
검색 조각 수: 13
검색 글 최대 길이: 128 토큰


### 2. 검색 조각을 임베딩하기

임베딩은 글의 의미를 숫자 목록으로 표현하는 작업입니다. 같은 모델로 질문과 자료를 바꿔야 비교할 수 있습니다. 이미 내려받은 모델을 CPU에서 사용합니다. 벡터 길이를 1로 맞추면 두 벡터의 곱으로 코사인 유사도를 계산할 수 있습니다.

In [2]:
# [프로젝트 추가 기록] SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
# [모델 추가] jhgan/ko-sroberta-multitask-mrl 선택 근거는 02번, 현재 설정은 src/car_search_rag/zzong_santafe_lag/config.py에 있습니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
import numpy as np
from sentence_transformers import SentenceTransformer

# 로컬 캐시에 있는 모델만 읽습니다. 외부 추론 API를 호출하지 않습니다.
embedding_model = SentenceTransformer(model_name, device="cpu", local_files_only=True)
assert embedding_model.max_seq_length == token_budget
assert all(token_count(chunk["content"]) <= embedding_model.max_seq_length for chunk in search_chunks)

chunk_vectors = embedding_model.encode(
    [chunk["content"] for chunk in search_chunks],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=False,
)
assert chunk_vectors.shape == (len(search_chunks), 768)
assert np.isfinite(chunk_vectors).all()
assert np.allclose(np.linalg.norm(chunk_vectors, axis=1), 1.0, atol=1e-5)

print("임베딩 배열 크기:", chunk_vectors.shape)
print("모델 이름:", model_name)
print("벡터 길이:", round(float(np.linalg.norm(chunk_vectors[0])), 3))
print("벡터 숫자 예시 5개:", np.round(chunk_vectors[0, :5], 4).tolist())

임베딩 배열 크기: (13, 768)
모델 이름: jhgan/ko-sroberta-multitask-mrl
벡터 길이: 1.0
벡터 숫자 예시 5개: [-0.043699998408555984, -0.03500000014901161, -0.02889999933540821, 0.044199999421834946, 0.03830000013113022]


### 3. 비슷한 조각을 찾고 원문 묶음으로 연결

같은 원문에서 나눈 조각이 여러 번 검색될 수 있습니다. 같은 원문 ID는 한 번만 표시합니다. 각 원문의 조각 중 가장 높은 점수를 그 원문의 순위에 사용합니다.

유사도는 답이 맞을 확률이 아닙니다. 실제 원문에 질문의 답이 있는지 확인해야 합니다.

In [3]:
# [프로젝트 추가 기록] SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
def rank_parents(query_vector, top_k=3):
    """질문 벡터로 조각을 검색하고, 원문 묶음을 중복 없이 높은 점수 순으로 돌려줍니다."""
    scores = chunk_vectors @ query_vector
    results = []
    seen_parent_ids = set()
    for index in np.argsort(-scores, kind="stable"):
        chunk = search_chunks[int(index)]
        parent_id = chunk["metadata"]["parent_record_id"]
        if parent_id in seen_parent_ids:
            continue
        seen_parent_ids.add(parent_id)
        results.append({
            "score": float(scores[index]),
            "matched_chunk": chunk,
            "parent_record": parent_lookup[parent_id],
        })
        if len(results) == top_k:
            break
    return results

def search_manual(question, top_k=3):
    """질문을 같은 모델로 임베딩해 원문과 그림 참조가 담긴 검색 후보를 가져옵니다."""
    if not question.strip():
        raise ValueError("질문을 입력하세요.")
    if token_count(question) > embedding_model.max_seq_length:
        raise ValueError("질문이 모델 입력 한도보다 깁니다. 질문을 짧게 나눠 주세요.")
    if not 1 <= top_k <= len(parent_lookup):
        raise ValueError("검색 결과 수는 원문 묶음 수 안에서 지정하세요.")
    query_vector = embedding_model.encode(
        question, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
    )
    return rank_parents(query_vector, top_k=top_k)

# 기대 원문 ID는 사람이 PDF를 확인하고 정한 평가 기준입니다.
evaluation_questions = [
    ("실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?", "overview_33"),
    ("후드 열림 레버의 위치를 보여주는 안내도를 찾아줘.", "overview_33"),
    ("지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요?", "overview_33"),
    ("유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘.", "overview_33"),
    ("시동 버튼 위치를 보여주는 안내도가 필요해.", "overview_34"),
    ("스마트폰 무선 충전 표시등은 내부 안내도의 어느 항목인가요?", "overview_34"),
    ("글로브 박스와 동승석 트레이의 위치를 찾아줘.", "overview_34"),
    ("주차 안전 버튼과 주차 뷰 버튼 위치가 나오는 안내도를 보여줘.", "overview_34"),
    ("와이퍼와 워셔 스위치는 내부 안내도 몇 번인가요?", "overview_35"),
    ("블루투스 핸즈프리 버튼과 음성 인식 버튼은 어디에 있나요?", "overview_35"),
    ("패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.", "overview_35"),
    ("차간거리 버튼과 차로 주행 보조 버튼 위치를 찾아줘.", "overview_35"),
    ("1.6 T-GDi HEV에 표시된 엔진 오일 SAE 점도는 무엇인가요?", "table_44"),
    ("0W-20이 표시된 엔진 오일 점도 표를 보여줘.", "table_44"),
    ("엔진 오일 SAE 점도 표의 온도 눈금은 어디부터 어디까지인가요?", "table_44"),
    ("안전벨트를 착용할 때 골반띠는 어디를 지나야 하나요?", "seatbelt_54"),
    ("안전벨트 버클을 얼마나 밀어 넣어야 하나요?", "seatbelt_54"),
    ("안전벨트 스토퍼를 쓰면 어떤 문제가 생기나요?", "seatbelt_54"),
    ("차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?", "seatbelt_54"),
    ("뒷좌석 놀이방 매트는 왜 안전벨트 사용에 문제가 되나요?", "seatbelt_54"),
]
assert all(token_count(question) <= token_budget for question, _ in evaluation_questions)

# 여러 질문을 함께 임베딩해 반복 실행 시간을 줄입니다.
question_vectors = embedding_model.encode(
    [question for question, _ in evaluation_questions],
    convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False,
)
evaluation_results = []
for (question, expected_id), vector in zip(evaluation_questions, question_vectors):
    hits = rank_parents(vector, top_k=3)
    ids = [hit["parent_record"]["metadata"]["record_id"] for hit in hits]
    evaluation_results.append({
        "question": question, "expected_id": expected_id, "actual_first_id": ids[0],
        "top1_correct": ids[0] == expected_id, "top3_correct": expected_id in ids,
        "top_score": hits[0]["score"],
    })

print("평가 질문 수:", len(evaluation_results))
print("첫 번째 결과가 기대한 원문:", sum(r["top1_correct"] for r in evaluation_results))
print("상위 세 결과에 기대한 원문 포함:", sum(r["top3_correct"] for r in evaluation_results))
for result in evaluation_results:
    status = "일치" if result["top1_correct"] else "다른 원문"
    print(f'{status} | 기대: {result["expected_id"]} | 첫 결과: {result["actual_first_id"]} | {result["question"]}')

평가 질문 수: 20
첫 번째 결과가 기대한 원문: 15
상위 세 결과에 기대한 원문 포함: 20
일치 | 기대: overview_33 | 첫 결과: overview_33 | 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요?
일치 | 기대: overview_33 | 첫 결과: overview_33 | 후드 열림 레버의 위치를 보여주는 안내도를 찾아줘.
일치 | 기대: overview_33 | 첫 결과: overview_33 | 지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요?
다른 원문 | 기대: overview_33 | 첫 결과: overview_34 | 유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘.
일치 | 기대: overview_34 | 첫 결과: overview_34 | 시동 버튼 위치를 보여주는 안내도가 필요해.
일치 | 기대: overview_34 | 첫 결과: overview_34 | 스마트폰 무선 충전 표시등은 내부 안내도의 어느 항목인가요?
일치 | 기대: overview_34 | 첫 결과: overview_34 | 글로브 박스와 동승석 트레이의 위치를 찾아줘.
일치 | 기대: overview_34 | 첫 결과: overview_34 | 주차 안전 버튼과 주차 뷰 버튼 위치가 나오는 안내도를 보여줘.
일치 | 기대: overview_35 | 첫 결과: overview_35 | 와이퍼와 워셔 스위치는 내부 안내도 몇 번인가요?
다른 원문 | 기대: overview_35 | 첫 결과: overview_34 | 블루투스 핸즈프리 버튼과 음성 인식 버튼은 어디에 있나요?
다른 원문 | 기대: overview_35 | 첫 결과: overview_34 | 패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.
다른 원문 | 기대: overview_35 | 첫 결과: overview_34 | 차간거리 버튼과 차로 주행 보조 버튼 위치를 찾아줘.
일치 | 기대: table_44 | 첫 결과: table_44 | 1.6 T-GDi HE

### 4. 원문과 실제 PDF 그림 가져오기

44쪽 표 질문의 검색 결과에서 원문과 이미지 정보를 가져옵니다. 파일로 아직 저장하지 않은 그림도 PDF 쪽수와 이름으로 읽을 수 있습니다.

아래 셀은 실제 그림을 읽어 크기를 확인합니다. 파일 저장은 하지 않습니다. retrieved_images는 이후 챗봇 화면에 표시할 수 있는 실제 이미지 객체 목록입니다.

In [4]:
# [프로젝트 추가 기록] SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
demo_question = "0W-20이 표시된 엔진 오일 점도 표를 보여줘."
demo_results = search_manual(demo_question)
top_hit = demo_results[0]
demo_parent = top_hit["parent_record"]

print("질문:", demo_question)
print("검색된 주제:", demo_parent["metadata"]["title"])
print("유사도:", round(top_hit["score"], 4))
print("PDF 출처:", demo_parent["metadata"]["source_pages"])
print("매뉴얼 쪽수:", demo_parent["metadata"]["manual_page_number"])
print("\n전체 검색용 원문:")
print(demo_parent["content"])

def resolve_pdf_image(image_part):
    """기록된 PDF 페이지에서 같은 이름의 그림을 읽습니다."""
    number = image_part["pdf_page_number"]
    image = next((item for item in reader.pages[number - 1].images if item.name == image_part["name"]), None)
    if image is None:
        raise ValueError(f'PDF {number}쪽에서 {image_part["name"]} 이미지를 찾지 못했습니다.')
    assert tuple(image.image.size) == tuple(image_part["size"])
    return image.image

retrieved_images = [resolve_pdf_image(part) for part in demo_parent["image_parts"]]
print("\n연결된 실제 그림 수:", len(retrieved_images))
for part, image in zip(demo_parent["image_parts"], retrieved_images):
    print("PDF", part["pdf_page_number"], "|", part["name"], "| 크기:", image.size)

assert demo_parent["metadata"]["record_id"] == "table_44"
assert len(retrieved_images) == 2
assert "0W-20" in demo_parent["content"]

질문: 0W-20이 표시된 엔진 오일 점도 표를 보여줘.
검색된 주제: 온도에 따른 엔진 오일 SAE 점도 분류표
유사도: 0.7546
PDF 출처: [44]
매뉴얼 쪽수: 39

전체 검색용 원문:
온도에 따른 엔진 오일 SAE 점도 분류표
SAE 점도와 사용 온도 범위
온도 (℃)
1.6 T-GDi HEV
[그림 설명] 온도 눈금은 -30도부터 50도까지 표시되어 있다. 1.6 T-GDi HEV 행에 0W-20이 표시되어 있다.

연결된 실제 그림 수: 2
PDF 44 | I1.jpg | 크기: (945, 71)
PDF 44 | I2.jpg | 크기: (945, 71)


### 5. 자료에 없는 질문도 점검하기

유사도 검색은 답이 없어도 비슷한 후보를 반환합니다. 다음 질문은 이번 작은 자료 묶음에 답이 없습니다. 차대번호·타이어 공기압은 전체 매뉴얼에는 있을 수 있으므로 이 실험에 없다는 뜻입니다.

유사도 숫자를 임의로 정답·거절 기준으로 정하지 않습니다. 최종 챗봇에서는 원문이 답을 뒷받침하지 못할 때 확인하기 어렵다고 안내하는 단계가 필요합니다.

In [5]:
# [프로젝트 추가 기록] SentenceTransformer 임베딩 개념을 프로젝트에서 선택한 jhgan 모델과 부모·자식 검색에 적용합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/car_search_rag/zzong_santafe_lag/learning_additions.md
missing_questions = [
    "오늘 서울 날씨와 내일 강수확률을 알려줘.",
    "차대번호는 차체의 어디에 새겨져 있나요?",
    "싼타페의 권장 타이어 공기압 수치를 알려줘.",
]
for question in missing_questions:
    hit = search_manual(question, top_k=1)[0]
    print("이번 자료에 답 없음 |", question)
    print("그래도 반환된 후보:", hit["parent_record"]["metadata"]["title"],
          "| 유사도:", round(hit["score"], 4))
print("\n이번 단계:", "검색 근거와 그림 연결 실험")
print("생성 답변:", "아직 만들지 않음")
print("벡터 파일 / Supabase 저장:", "실행하지 않음")

이번 자료에 답 없음 | 오늘 서울 날씨와 내일 강수확률을 알려줘.
그래도 반환된 후보: 온도에 따른 엔진 오일 SAE 점도 분류표 | 유사도: 0.2921
이번 자료에 답 없음 | 차대번호는 차체의 어디에 새겨져 있나요?
그래도 반환된 후보: 차량 내부 II | 유사도: 0.6392
이번 자료에 답 없음 | 싼타페의 권장 타이어 공기압 수치를 알려줘.
그래도 반환된 후보: 온도에 따른 엔진 오일 SAE 점도 분류표 | 유사도: 0.5489

이번 단계: 검색 근거와 그림 연결 실험
생성 답변: 아직 만들지 않음
벡터 파일 / Supabase 저장: 실행하지 않음


### 이번 실행 결과와 다음 개선

이번 20개 질문에서 첫 결과의 원문 일치는 15개, 상위 세 원문에 기대한 원문이 포함된 경우는 20개였습니다. 원문 묶음 5개만 사용한 작은 실험이며, 답변 정확도나 전체 PDF 성능을 측정한 결과는 아닙니다.

유리창 잠금·블루투스/음성 인식·패들 쉬프트·차간거리/차로 주행 보조·차단 클립 질문은 첫 원문이 달랐습니다. 평가 질문과 결과를 그대로 남겼습니다. 다음에는 원문에 정확한 부품 이름이 있는지 확인하는 검색과 의미 검색을 함께 비교할 수 있습니다.

자료에 없는 차대번호 질문도 0.6392의 후보를 반환했습니다. 유사도만으로 답변 가능 여부를 판단하지 않으며, 원문에서 실제 근거를 확인하는 단계가 필요합니다.